# Importing Lib

In [1]:
from tensorflow.keras.layers import Dense, Flatten 
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm  

In [ ]:
def load_image_csv_efficiently(filepath, chunk_size=1000):
    # Read only first row to get column count
    first_row = pd.read_csv(filepath, nrows=1)
    total_columns = len(first_row.columns)
    
    # Initialize arrays
    all_labels = []
    all_pixels = []
    
    # Read CSV in chunks with progress bar
    chunks = pd.read_csv(filepath, chunksize=chunk_size,memory_map=True)
    for chunk in tqdm(chunks, desc="Loading data"):
        # Split labels and pixels
        labels = chunk.iloc[:, 0].values
        pixels = chunk.iloc[:, 1:].values.astype('float32')
        
        all_labels.append(labels)
        all_pixels.append(pixels)
    
    # Concatenate chunks
    Y_raw = np.concatenate(all_labels)
    X_flat = np.concatenate(all_pixels)
    
    return Y_raw, X_flat

In [4]:
Y_raw, X_flat = load_image_csv_efficiently(r'M:\University\University Work\Semestor 5\Ann & DL\LAB\ANN & DL\Data\Images224.csv')
np.savez_compressed('images224.npz', X=X_flat, Y=Y_raw)

Loading data: 3it [13:06, 262.08s/it]


In [14]:
expected = 224 * 224 * 3
if X_flat.shape[1] != expected:
    raise ValueError(f"Expected {expected} columns for image pixels, found {X_flat.shape[1]}")

X = X_flat.reshape(-1, 224, 224, 3) / 255.0  # normalize to [0,1]


In [15]:
# encode labels and convert to one-hot
le = LabelEncoder()
Y_enc = le.fit_transform(Y_raw)
n_classes = len(np.unique(Y_enc))
Y = to_categorical(Y_enc, n_classes)


In [16]:
# train/test split (stratify to keep class balance)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=30, stratify=Y_enc)

# Transfer Learning & ResNet (Using other weights and using them for our dataset) 

In [17]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
# only using one top layer input layer from resent 50 layers
for layer in base_model.layers:
    layer.trainable = False  # frezzing the 49 layers 

x = base_model.output   
x = Flatten()(x)
x = Dense(200, activation='relu')(x)
predictions = Dense(3, activation='softmax')(x)  # 3 classes
Resnet_model = Model(inputs=base_model.input, outputs=predictions)  # combining the model and resent50
Resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
Resnet_model.summary()
Resnet_model.fit(X_train, Y_train, epochs=10, batch_size=32, validation_data=(X_test, Y_test))
Resnet_model.save(r"M:\University\University Work\Semestor 5\Ann & DL\LAB\ANN & DL\Model\Resnet50_224_model.keras")

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 43,658,915 (166.55 MB)

 Trainable params: 20,071,203 (76.57 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 168s 2s/step - accuracy: 0.6752 - loss: 2.0020 - val_accuracy: 0.6539 - val_loss: 0.6150
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 335s 5s/step - accuracy: 0.9003 - loss: 0.2821 - val_accuracy: 0.9437 - val_loss: 0.1980
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 138s 2s/step - accuracy: 0.9436 - loss: 0.1833 - val_accuracy: 0.9356 - val_loss: 0.1921
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.9612 - loss: 0.1253 - val_accuracy: 0.9497 - val_loss: 0.1337
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.9502 - loss: 0.1338 - val_accuracy: 0.9235 - val_loss: 0.2264
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.9386 - loss: 0.1488 - val_accuracy: 0.9658 - val_loss: 0.1153
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 132s 2s/step - accuracy: 0.9814 - loss: 0.0675 - val_accuracy: 0.9859 - val_loss: 0.0580
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 133s 2s/step - accuracy: 0.9854 - loss: 0.0548 - val_accuracy: 0.9618 - v

In [18]:
loss, accuracy = Resnet_model.evaluate(X_test, Y_test)

print(f'Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}')

16/16 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - accuracy: 0.8833 - loss: 0.2865
Test Loss: 0.2865, Test Accuracy: 0.8833
